# NB2 — Core-7 Drop & Category-Clean Positives

Notebook này thực hiện **chỉ phần category drop trước negative sampling**:

1. dùng official split `train/valid/test` của Polyvore1000;
2. map `master_category` thành `TOP/BOTTOM/DRESS/OUTERWEAR/SHOES/BAG/HAT/DROP`;
3. loại item được map thành `DROP`;
4. đếm lại số item của outfit;
5. giữ outfit còn ít nhất 3 item;
6. xuất **category-clean positive JSONL**.

> Đây là output trung gian của bước category drop, chưa phải dataset `READY_TO_TRAIN`. Sau đó vẫn phải kiểm tra ảnh/embedding, drop item lỗi và đếm lại độ dài outfit. Notebook không tạo negative; negative cũ không được dùng lại sau khi drop vì item index và outfit composition có thể đã thay đổi.

## Luồng xử lý

```text
Polyvore1000 positive outfits
        ↓
master_category → Core-7 hoặc DROP
        ↓
loại item DROP
        ↓
recompute outfit length
        ↓
giữ outfit length >= 3
        ↓
category-clean positive JSONL
        ↓
image/embedding validation
        ↓
final clean positives
        ↓
teammate tạo negative mới
```

Logic chính nằm trong `src/data/prepare_core7_dataset.py`. Notebook chỉ cấu hình, gọi hàm và hiển thị kết quả.

## 1. Cài thư viện

Chỉ cần `datasets` để đọc Polyvore1000 từ Hugging Face. Không chạy FashionCLIP ở notebook này vì cache embedding hiện tại có thể được reuse sau khi lọc item.

In [ ]:
%pip install -q "datasets>=3,<5"

## 2. Lấy repository và import `src/`

Trên Colab, notebook sẽ clone repo nếu `/content/opisoverated` chưa tồn tại. Nếu đang chạy ngay trong repo local, code sẽ dùng repo hiện tại.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys


REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"


def find_repo_root(start: Path = Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
    return None


REPO_ROOT = find_repo_root()

if REPO_ROOT is None:
    REPO_ROOT = Path("/content/opisoverated")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.prepare_core7_dataset import (
    CORE_CATEGORIES,
    load_category_mapping,
    prepare_clean_positive_split,
)

print("Repo root:", REPO_ROOT)
print("Core categories:", CORE_CATEGORIES)

## 3. Mount Google Drive và cấu hình output

Output mặc định được ghi vào Drive của bạn:

```text
MyDrive/fashion_audit/core7_drop_v1/
```

`DEBUG_LIMIT=50` nghĩa là lần chạy thử chỉ xử lý 50 outfit đầu của split train.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/fashion_audit/core7_drop_v1")
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MAPPING_PATH = REPO_ROOT / "configs/category_mapping_core7_v1.json"
MIN_ITEMS = 3

# Chạy thử an toàn trước. Đặt None chỉ trong cell full run ở cuối.
DEBUG_SPLIT = "train"
DEBUG_LIMIT = 50

print("Mapping:", MAPPING_PATH)
print("Drive output:", DRIVE_OUTPUT_ROOT)

## 4. Kiểm tra mapping Core-7/DROP

Mapping hiện là **draft**. Nó chứa quyết định cho toàn bộ master category hiện có. Category không thuộc quần áo chính được map thành `DROP`.

Cell này chỉ đọc và thống kê; chưa lọc dữ liệu.

In [ ]:
mapping_metadata, category_mapping = load_category_mapping(MAPPING_PATH)

decision_counts = {}
for decision in category_mapping.values():
    decision_counts[decision] = decision_counts.get(decision, 0) + 1

print("Mapping version:", mapping_metadata["mapping_version"])
print("Mapping status :", mapping_metadata["status"])
print("Master categories:", len(category_mapping))
print("Decision counts:")
for decision, count in sorted(decision_counts.items()):
    print(f"  {decision:10s}: {count}")

print("\nVí dụ category được giữ:")
shown = 0
for master, coarse in category_mapping.items():
    if coarse != "DROP":
        print(f"  {master:35s} -> {coarse}")
        shown += 1
        if shown == 20:
            break

## 5. Chạy thử trên 50 outfit train

Trong bước này, core code sẽ:

- load item/kit metadata của official train split;
- kiểm tra mapping cover toàn bộ master category;
- drop item theo mapping;
- tạo positive sample canonical;
- drop outfit còn dưới 3 item;
- validate output;
- ghi JSONL debug vào Drive.

In [ ]:
DEBUG_OUTPUT = DRIVE_OUTPUT_ROOT / "debug_category_clean_train.jsonl"

debug_report = prepare_clean_positive_split(
    split=DEBUG_SPLIT,
    output_path=DEBUG_OUTPUT,
    mapping_path=MAPPING_PATH,
    min_items=MIN_ITEMS,
    debug_limit=DEBUG_LIMIT,
)

debug_report

## 6. Xem sample sau khi drop

Positive output phải có:

- `sample_id = <kit_id>_pos`;
- `source_kit_id`;
- `paired_positive_sample_id = null`;
- chỉ còn item thuộc Core-7;
- `label = 1`;
- `negative_metadata = null`.

In [ ]:
def read_jsonl_head(path: Path, n: int = 5):
    rows = []
    with path.open("r", encoding="utf-8") as stream:
        for index, line in enumerate(stream):
            if index >= n:
                break
            rows.append(json.loads(line))
    return rows


for sample in read_jsonl_head(DEBUG_OUTPUT, n=5):
    print(json.dumps(sample, ensure_ascii=False, indent=2))
    print("-" * 80)

## 7. Đọc nhanh ý nghĩa report

Các field quan trọng:

- `raw_item_count`: số item trước drop;
- `kept_item_count`: số item thuộc Core-7;
- `dropped_item_count`: số item bị loại;
- `outfits_kept`: số positive còn ít nhất 3 item;
- `outfits_dropped_below_min_items`: số outfit bị loại;
- `validation.pass`: phải là `true`.

Debug chỉ lấy 50 outfit nhưng item statistics vẫn được tính trên metadata của cả split, vì mapping phải được kiểm tra đầy đủ.

In [ ]:
summary = {
    "items_before_drop": debug_report["filter"]["raw_item_count"],
    "items_after_drop": debug_report["filter"]["kept_item_count"],
    "items_dropped": debug_report["filter"]["dropped_item_count"],
    "debug_outfits_processed": debug_report["outfits"]["kits_processed"],
    "debug_outfits_kept": debug_report["outfits"]["outfits_kept"],
    "debug_outfits_dropped": debug_report["outfits"]["outfits_dropped_below_min_items"],
    "validation_pass": debug_report["validation"]["pass"],
}

print(json.dumps(summary, ensure_ascii=False, indent=2))

## 8. Full run — chỉ bật sau khi nhóm review mapping

Mặc định `RUN_FULL=False` để tránh vô tình tạo lại toàn bộ dataset.

Sau khi nhóm review `configs/category_mapping_core7_v1.json`:

1. đổi `status` từ `draft` thành `frozen`;
2. đổi `RUN_FULL=True`;
3. chạy cell này.

Output gồm category-clean positives của ba official split và report tương ứng. Đây chưa phải `clean_*.jsonl` cuối cùng nếu chưa qua image/embedding validation.

In [ ]:
RUN_FULL = False

if RUN_FULL:
    mapping_metadata, _ = load_category_mapping(MAPPING_PATH)
    if mapping_metadata.get("status") != "frozen":
        raise RuntimeError(
            "Mapping vẫn là draft. Hãy review mapping và đổi status='frozen' trước full run."
        )

    full_reports = {}
    for split in ("train", "valid", "test"):
        output_path = DRIVE_OUTPUT_ROOT / f"category_clean_{split}.jsonl"
        report_path = DRIVE_OUTPUT_ROOT / f"category_clean_{split}_report.json"

        report = prepare_clean_positive_split(
            split=split,
            output_path=output_path,
            mapping_path=MAPPING_PATH,
            min_items=MIN_ITEMS,
            debug_limit=None,
        )
        with report_path.open("w", encoding="utf-8") as stream:
            json.dump(report, stream, ensure_ascii=False, indent=2)
            stream.write("\n")

        full_reports[split] = report

    print("FULL RUN COMPLETE")
    for split, report in full_reports.items():
        print(
            split,
            "| kept:", report["outfits"]["outfits_kept"],
            "| dropped:", report["outfits"]["outfits_dropped_below_min_items"],
        )
else:
    print("RUN_FULL=False — mới chỉ chạy debug, chưa tạo full category-clean positives.")

## 9. Việc tiếp theo sau notebook này

Notebook này hoàn thành bước **category drop** khi nhóm có:

```text
category_clean_train.jsonl
category_clean_valid.jsonl
category_clean_test.jsonl
```

Tiếp theo:

1. kiểm tra/decode ảnh hoặc đối chiếu FashionCLIP embedding cache;
2. drop item thiếu ảnh/embedding, đếm lại outfit và giữ length >= 3;
3. xuất final `clean_train/valid/test.jsonl`;
4. negative generator đọc từng final clean split;
5. thay đúng một item bằng replacement cùng `master_category` và cùng split;
6. tạo lại `swapped_item_index` theo outfit đã clean;
7. merge positive + negative thành scorer dataset;
8. train/evaluate Type-aware scorer.

> Không dùng lại negative JSONL cũ sau khi drop.